# Science visit pairs at fixed azimuth and rotator, separated in elevation (v1)

**Author:** Aaron Roodman  
**Date Created:** 2026-10-29  
**Last Modified:** 2026-10-29  
**Status:** In Progress  
**Keywords:** elevation, look-up table, LUT, bounce test, science visits, optical state

## Description

Finds same-night pairs of science visits that move at least 10 deg in elevation while
azimuth and camera rotator angle each change by less than 5 deg and the band is
unchanged. Such pairs are a science-survey analogue of the elevation bounce test, which
compares the optical state at two elevations directly.

1. Reads science visits (band, image type, pointing) from ConsDB through the
   `value_added` product, with the hexapod look-up table (LUT) per visit.
2. Forms every qualifying pair within each night, then keeps pairs whose two visits are
   at most `MAX_DT_MIN` apart, optionally only consecutive science visits.
3. Attaches the change in hexapod LUT and in the recovered open-loop optical state
   (degree of freedom, DOF) across each pair.

**Output:** `pairs.parquet` (all columns), `pairs.csv` (pointing and hexapod columns) and
`elevation_pairs.pdf`, in the dated run directory.

**Based on:** `aos/bounce` (elevation bounce test), `aos/cwfs_lut` (survey trends)

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-10-29 | Aaron Roodman | Initial version |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Pair Search](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [1]:
from rubinwork.common import study_run

DAY_OBS_RANGE = None          # (lo, hi) as YYYYMMDD, or None for every night
MIN_DELTA_EL_DEG = 10.0       # minimum |elevation change| [deg]
MAX_DELTA_AZ_DEG = 5.0        # maximum |azimuth change| [deg], wrapped to +-180
MAX_DELTA_ROT_DEG = 5.0       # maximum |camera rotator angle change| [deg]
MAX_DT_MIN = 2.0              # maximum time between the two visit starts [min]
CONSECUTIVE_ONLY = True       # keep only pairs of consecutive science visits
STATE_VARIANT = 'v50_34__miw__consdb_v1'   # recovered optical state to difference

run = study_run('aos/elevation_pairs', 'pairs_dt2min')

<a id='setup'></a>
## Setup & Imports

In [2]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

from rubinwork.common.utils import setup_plotting
from rubinwork.products import value_added as efd_db

warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)
setup_plotting()

<a id='functions'></a>
## Helper Functions

In [3]:
#: DOF labels in ts_ofc order: M2 hexapod 0-4, camera hexapod 5-9, then the bending modes.
HEX_LABELS = ['M2 dz', 'M2 dx', 'M2 dy', 'M2 rx', 'M2 ry',
              'Cam dz', 'Cam dx', 'Cam dy', 'Cam rx', 'Cam ry']
#: Units of the hexapod DOF: translations in um, tilts in deg.
HEX_UNITS = ['um', 'um', 'um', 'deg', 'deg'] * 2


def find_pairs(sci, min_del, max_daz, max_drot, max_dt_min, consecutive_only):
    """Pair science visits within each night that change elevation at fixed pointing.

    Parameters
    ----------
    sci : `pandas.DataFrame`
        One row per science visit with ``day_obs``, ``seq_num``, ``visit_id``, ``band``,
        ``obs_start_mjd`` [d], ``altitude_deg`` [deg], ``azimuth_deg_consdb`` [deg] and
        ``rotator_angle_deg`` [deg].
    min_del : `float`
        Minimum |elevation change| [deg].
    max_daz, max_drot : `float`
        Maximum |azimuth change| and |camera rotator angle change| [deg].
    max_dt_min : `float`
        Maximum time between the two visit starts [min].
    consecutive_only : `bool`
        Keep only pairs whose visits are consecutive among the night's science visits.

    Returns
    -------
    pairs : `pandas.DataFrame`
        One row per pair, earlier visit ``_1`` and later ``_2``, with ``d_el_deg``,
        ``d_az_deg``, ``d_rot_deg`` (later minus earlier, [deg]) and ``dt_min`` [min].
    """
    rows = []
    for day_obs, g in sci.groupby('day_obs'):
        g = g.sort_values('obs_start_mjd').reset_index(drop=True)
        t = g['obs_start_mjd'].to_numpy() * 1440.0
        el = g['altitude_deg'].to_numpy()
        az = g['azimuth_deg_consdb'].to_numpy()
        rot = g['rotator_angle_deg'].to_numpy()
        band = g['band'].to_numpy()
        i, j = np.triu_indices(len(g), 1)
        if consecutive_only:
            keep = j == i + 1
            i, j = i[keep], j[keep]
        d_az = (az[j] - az[i] + 180.0) % 360.0 - 180.0
        ok = ((np.abs(el[j] - el[i]) >= min_del) & (np.abs(d_az) < max_daz)
              & (np.abs(rot[j] - rot[i]) < max_drot) & (band[i] == band[j])
              & (t[j] - t[i] <= max_dt_min))
        for a, b, daz in zip(i[ok], j[ok], d_az[ok]):
            rows.append({'day_obs': day_obs,
                         'seq_num_1': g.at[a, 'seq_num'], 'seq_num_2': g.at[b, 'seq_num'],
                         'visit_id_1': g.at[a, 'visit_id'], 'visit_id_2': g.at[b, 'visit_id'],
                         'band': band[a], 'science_program': g.at[a, 'science_program'],
                         'el_1_deg': el[a], 'el_2_deg': el[b], 'd_el_deg': el[b] - el[a],
                         'az_1_deg': az[a], 'd_az_deg': daz,
                         'rot_1_deg': rot[a], 'd_rot_deg': rot[b] - rot[a],
                         'dt_min': t[b] - t[a]})
    return pd.DataFrame(rows)


def attach_deltas(pairs, per_visit, cols, prefix):
    """Add per-visit values at both ends of each pair and their change (2 minus 1).

    Parameters
    ----------
    pairs : `pandas.DataFrame`
        Output of `find_pairs`.
    per_visit : `pandas.DataFrame`
        Indexed by ``visit_id``, carrying `cols`.
    cols : `list` [`str`]
        Columns to difference.
    prefix : `str`
        Prefix for the output columns ``<prefix><col>_1`` and ``d_<prefix><col>``.

    Returns
    -------
    pairs : `pandas.DataFrame`
    """
    v1 = per_visit.reindex(pairs['visit_id_1'])[cols].to_numpy()
    v2 = per_visit.reindex(pairs['visit_id_2'])[cols].to_numpy()
    new = {}
    for k, c in enumerate(cols):
        new[f'{prefix}{c}_1'] = v1[:, k]
        new[f'd_{prefix}{c}'] = v2[:, k] - v1[:, k]
    return pd.concat([pairs, pd.DataFrame(new, index=pairs.index)], axis=1)

<a id='data'></a>
## Data Access

Science visits with band and pointing from ConsDB, plus the hexapod LUT from the `value_added` database.

In [4]:
lut_cols = [f'lut_dof{k}' for k in range(10)]
vis = efd_db.visits(day_obs_range=DAY_OBS_RANGE,
                    columns=['visit_id', 'day_obs', 'seq_num'] + lut_cols)
vis = efd_db.join_consdb(vis, groups=('meta',))
sci = vis[vis['img_type'] == 'science'].dropna(
    subset=['band', 'obs_start_mjd', 'altitude_deg', 'azimuth_deg_consdb', 'rotator_angle_deg'])
print(f'{len(sci)} science visits with band and pointing, {sci["day_obs"].nunique()} nights, '
      f'day_obs {sci["day_obs"].min()} to {sci["day_obs"].max()}')

        Use pytest instead. [astropy.tests.runner]
        Use pytest instead.


        Use pytest instead. [astropy.utils.decorators]
        Use pytest instead.


95938 science visits with band and pointing, 257 nights, day_obs 20250415 to 20260713


<a id='analysis'></a>
## Pair Search

All-pairs counts first, for scale, then the selected sample.

In [5]:
scan = find_pairs(sci, MIN_DELTA_EL_DEG, MAX_DELTA_AZ_DEG, MAX_DELTA_ROT_DEG,
                  max_dt_min=np.inf, consecutive_only=False)
print('All same-night pairs passing the pointing and band cuts, by time limit:')
print(f'{"max dt [min]":>13} {"pairs":>8} {"consecutive":>12} {"nights":>7}')
consec = find_pairs(sci, MIN_DELTA_EL_DEG, MAX_DELTA_AZ_DEG, MAX_DELTA_ROT_DEG,
                    max_dt_min=np.inf, consecutive_only=True)
for lim in [1, 2, 5, 10, 30, 60, np.inf]:
    print(f'{lim:>13} {(scan["dt_min"] <= lim).sum():>8} {(consec["dt_min"] <= lim).sum():>12} '
          f'{scan.loc[scan["dt_min"] <= lim, "day_obs"].nunique():>7}')

pairs = consec if CONSECUTIVE_ONLY else scan
pairs = pairs[pairs['dt_min'] <= MAX_DT_MIN].reset_index(drop=True)
print(f'\nSelected: {len(pairs)} pairs over {pairs["day_obs"].nunique()} nights '
      f'(dt <= {MAX_DT_MIN} min, consecutive only = {CONSECUTIVE_ONLY})')

All same-night pairs passing the pointing and band cuts, by time limit:
 max dt [min]    pairs  consecutive  nights


            1       81           81      50
            2      384           85     102
            5     5812           87     175
           10    19325           88     187
           30    45507           89     192
           60    58740           89     194
          inf   124020           89     197

Selected: 85 pairs over 52 nights (dt <= 2.0 min, consecutive only = True)


Attach the change across each pair in the hexapod LUT and in the recovered open-loop optical state (`dof*_olr`, the total state with the commanded corrections removed).

In [6]:
pairs = attach_deltas(pairs, sci.set_index('visit_id'), lut_cols, '')

st = efd_db.optical_state(STATE_VARIANT, day_obs_range=DAY_OBS_RANGE, wide=True, ok_only=True)
olr_cols = [f'dof{k}_olr' for k in range(50)]
pairs = attach_deltas(pairs, st.set_index('visit_id'), olr_cols + ['fwhm_cwfs_arcsec'], '')
have_state = pairs['d_dof0_olr'].notna()
pairs['has_state'] = have_state
print(f'{have_state.sum()} of {len(pairs)} pairs have a {STATE_VARIANT} state at both visits')

63 of 85 pairs have a v50_34__miw__consdb_v1 state at both visits


<a id='results'></a>
## Results & Plots

In [7]:
show = ['day_obs', 'seq_num_1', 'seq_num_2', 'band', 'el_1_deg', 'el_2_deg', 'd_el_deg',
        'd_az_deg', 'd_rot_deg', 'dt_min', 'has_state']
pd.set_option('display.width', 200)
print('Angles [deg], dt [min]')
print(pairs[show].round(2).to_string(max_rows=60))

print('\nPairs per band:')
print(pairs['band'].value_counts().to_string())
print('\n|elevation change| [deg]:')
print(pd.cut(pairs['d_el_deg'].abs(), [10, 12, 15, 20, 30, 90]).value_counts().sort_index().to_string())

Angles [deg], dt [min]
     day_obs  seq_num_1  seq_num_2 band  el_1_deg  el_2_deg  d_el_deg  d_az_deg  d_rot_deg  dt_min  has_state
0   20250609        429        430    z     30.44     44.23     13.79      2.05       4.48    0.86      False
1   20250701        543        544    i     49.03     59.91     10.88      2.47       1.31    0.77      False
2   20250710        392        393    z     31.38     41.40     10.02     -3.59       0.08    0.67      False
3   20250710        396        397    z     42.10     53.16     11.05      4.74       0.05    0.68      False
4   20250710        400        401    z     59.14     25.41    -33.73      3.07       0.08    0.88      False
5   20250710        483        484    z     30.44     42.77     12.32     -1.31      -0.08    0.69      False
6   20250713        630        631    r     57.23     68.02     10.78      4.37       0.17    0.77      False
7   20250718        304        305    r     48.32     68.76     20.44     -0.77      -2.16    0.9

In [8]:
# Change in hexapod DOF per degree of elevation, for the pairs with a recovered state.
# The bounce test reports the same quantity; a robust median over pairs is the first look.
s = pairs[pairs['has_state']]
rows = []
for k, (lab, unit) in enumerate(zip(HEX_LABELS, HEX_UNITS)):
    slope_olr = s[f'd_dof{k}_olr'] / s['d_el_deg']
    slope_lut = s[f'd_lut_dof{k}'] / s['d_el_deg']
    rows.append({'DOF': f'dof{k} {lab}', 'unit': f'{unit}/deg',
                 'median dOLR/dEl': slope_olr.median(),
                 'nMAD dOLR/dEl': 1.4826 * (slope_olr - slope_olr.median()).abs().median(),
                 'median dLUT/dEl': slope_lut.median(), 'n pairs': slope_olr.notna().sum()})
hex_table = pd.DataFrame(rows)
print('Hexapod change per degree of elevation (later minus earlier visit), per pair:')
print(hex_table.round(4).to_string(index=False))

Hexapod change per degree of elevation (later minus earlier visit), per pair:
        DOF    unit  median dOLR/dEl  nMAD dOLR/dEl  median dLUT/dEl  n pairs
 dof0 M2 dz  um/deg           3.4184        20.0224           0.0523       63
 dof1 M2 dx  um/deg          -6.5924        37.0903          -0.0000       63
 dof2 M2 dy  um/deg         -21.6667        29.2223          -0.5778       63
 dof3 M2 rx deg/deg          -0.0001         0.0003          -0.0000       63
 dof4 M2 ry deg/deg           0.0001         0.0004           0.0000       63
dof5 Cam dz  um/deg          -3.8953        18.8678           0.0524       63
dof6 Cam dx  um/deg          13.6283        39.1336          -0.3879       63
dof7 Cam dy  um/deg          14.6241        36.2704          -0.3870       63
dof8 Cam rx deg/deg           0.0001         0.0001          -0.0000       63
dof9 Cam ry deg/deg          -0.0000         0.0001           0.0000       63


In [9]:
pairs.to_parquet(run.path / 'pairs.parquet', index=False)
csv_cols = show + ['visit_id_1', 'visit_id_2', 'science_program', 'az_1_deg', 'rot_1_deg'] \
    + [f'd_lut_dof{k}' for k in range(10)] + [f'd_dof{k}_olr' for k in range(10)]
pairs[csv_cols].to_csv(run.path / 'pairs.csv', index=False, float_format='%.5g')
hex_table.to_csv(run.path / 'hexapod_slopes.csv', index=False, float_format='%.5g')

with PdfPages(run.path / 'elevation_pairs.pdf') as pdf:
    fig = plt.figure(figsize=(8.5, 6))
    fig.text(0.05, 0.95, 'Science visit pairs: elevation change at fixed azimuth and rotator',
             fontsize=13, va='top')
    txt = (f'Cuts: |dEl| >= {MIN_DELTA_EL_DEG} deg, |dAz| < {MAX_DELTA_AZ_DEG} deg, '
           f'|dRot| < {MAX_DELTA_ROT_DEG} deg, same band, dt <= {MAX_DT_MIN} min, '
           f'consecutive only = {CONSECUTIVE_ONLY}\n'
           f'Science visits searched: {len(sci)} over {sci["day_obs"].nunique()} nights\n'
           f'Pairs: {len(pairs)} over {pairs["day_obs"].nunique()} nights; '
           f'{pairs["has_state"].sum()} with recovered state ({STATE_VARIANT})\n\n'
           'Hexapod change per degree of elevation (median over pairs):\n'
           + hex_table.round(4).to_string(index=False))
    fig.text(0.05, 0.88, txt, fontsize=8, va='top', family='monospace')
    pdf.savefig(fig)
    plt.close(fig)

    fig, ax = plt.subplots(2, 2, figsize=(11, 8))
    ax[0, 0].hist(pairs['d_el_deg'], bins=40)
    ax[0, 0].set_xlabel('elevation change, later minus earlier [deg]')
    ax[0, 0].set_ylabel('pairs')
    ax[0, 1].hist(pairs['dt_min'], bins=40)
    ax[0, 1].set_xlabel('time between visit starts [min]')
    ax[1, 0].scatter(pairs['el_1_deg'], pairs['el_2_deg'], s=6, c=pairs['rot_1_deg'],
                     cmap='coolwarm', vmin=-80, vmax=80)
    ax[1, 0].set_xlabel('elevation, earlier visit [deg]')
    ax[1, 0].set_ylabel('elevation, later visit [deg]')
    ax[1, 1].scatter(pairs['az_1_deg'], pairs['rot_1_deg'], s=6)
    ax[1, 1].set_xlabel('azimuth, earlier visit [deg]')
    ax[1, 1].set_ylabel('camera rotator angle, earlier visit [deg]')
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    fig, ax = plt.subplots(figsize=(11, 4))
    per_night = pairs.groupby('day_obs').size()
    ax.bar(np.arange(len(per_night)), per_night.to_numpy())
    step = max(1, len(per_night) // 20)
    ax.set_xticks(np.arange(len(per_night))[::step])
    ax.set_xticklabels(per_night.index[::step], rotation=90, fontsize=7)
    ax.set_ylabel('pairs per night')
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    fig, axs = plt.subplots(2, 5, figsize=(15, 6))
    for k, a in enumerate(axs.flat):
        a.scatter(s['d_el_deg'], s[f'd_dof{k}_olr'], s=6, label='open-loop state')
        a.scatter(s['d_el_deg'], s[f'd_lut_dof{k}'], s=6, label='LUT')
        a.set_title(f'dof{k} {HEX_LABELS[k]}', fontsize=9)
        a.set_xlabel('elevation change [deg]', fontsize=8)
        a.set_ylabel(f'change [{HEX_UNITS[k]}]', fontsize=8)
    axs[0, 0].legend(fontsize=7)
    fig.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)
print(f'Wrote {run.path}')

/lscratch/roodman/ipykernel_1648413/4231815813.py:36: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


/lscratch/roodman/ipykernel_1648413/4231815813.py:47: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


/lscratch/roodman/ipykernel_1648413/4231815813.py:59: UserWarning: The figure layout has changed to tight
  fig.tight_layout()


Wrote /sdf/group/rubin/u/roodman/LSST/rubin-work/studies/aos/elevation_pairs/20261029_pairs_dt2min
